# Data Preparation

In [1]:
# Raw Dataset

In [2]:
import os
import pandas as pd
import json
import glob

cloudtrail_folder = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Raw/CloudTrail/CloudTrail/*.json")

records = []

print("Loading CloudTrail logs...")

for file in glob.glob(cloudtrail_folder):
    with open(file, "r") as f:
        data = json.load(f)

        # Extract records safely
        if "Records" in data:
            records.extend(data["Records"])

print("Total raw records:", len(records))

df_identity = pd.DataFrame(records)

# limit to 1000 rows for now
df_identity = df_identity.head(1000)

print("Data loaded successfully")
print("Shape:", df_identity.shape)

Loading CloudTrail logs...
Total raw records: 2900
Data loaded successfully
Shape: (1000, 27)


## Data Inspection

In [3]:
print(df_identity.columns)

Index(['eventVersion', 'userIdentity', 'eventTime', 'eventSource', 'eventName',
       'awsRegion', 'sourceIPAddress', 'userAgent', 'requestParameters',
       'responseElements', 'requestID', 'eventID', 'readOnly', 'eventType',
       'managementEvent', 'recipientAccountId', 'eventCategory', 'tlsDetails',
       'resources', 'errorCode', 'errorMessage', 'additionalEventData',
       'sharedEventID', 'apiVersion', 'vpcEndpointId',
       'sessionCredentialFromConsole', 'serviceEventDetails'],
      dtype='str')


In [4]:
df_identity.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 27 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   eventVersion                  1000 non-null   str   
 1   userIdentity                  1000 non-null   object
 2   eventTime                     1000 non-null   str   
 3   eventSource                   1000 non-null   str   
 4   eventName                     1000 non-null   str   
 5   awsRegion                     1000 non-null   str   
 6   sourceIPAddress               1000 non-null   str   
 7   userAgent                     1000 non-null   str   
 8   requestParameters             927 non-null    object
 9   responseElements              142 non-null    object
 10  requestID                     999 non-null    str   
 11  eventID                       1000 non-null   str   
 12  readOnly                      1000 non-null   bool  
 13  eventType                     

In [5]:
# userIdentity         object
# requestParameters    object
# resources            object
# additionalEventData  object

# These must be flattened.

In [6]:
df_identity[['eventTime','eventSource','eventName','sourceIPAddress']].head()

,eventTime,eventSource,eventName,sourceIPAddress
0,2023-07-10T12:06:32Z,iam.amazonaws.com,CreateRole,192.168.10.20
1,2023-07-10T12:06:32Z,iam.amazonaws.com,GetRole,192.168.10.20
2,2023-07-10T12:06:32Z,iam.amazonaws.com,ListRolePolicies,192.168.10.20
3,2023-07-10T12:06:32Z,iam.amazonaws.com,ListAttachedRolePolicies,192.168.10.20
4,2023-07-10T12:06:35Z,ec2.amazonaws.com,AssociateRouteTable,192.168.10.20


In [7]:
print(df_identity.columns)

Index(['eventVersion', 'userIdentity', 'eventTime', 'eventSource', 'eventName',
       'awsRegion', 'sourceIPAddress', 'userAgent', 'requestParameters',
       'responseElements', 'requestID', 'eventID', 'readOnly', 'eventType',
       'managementEvent', 'recipientAccountId', 'eventCategory', 'tlsDetails',
       'resources', 'errorCode', 'errorMessage', 'additionalEventData',
       'sharedEventID', 'apiVersion', 'vpcEndpointId',
       'sessionCredentialFromConsole', 'serviceEventDetails'],
      dtype='str')


In [8]:
df_identity['userIdentity'].iloc[0]

{'type': 'IAMUser',
 'principalId': 'AIDATFQR7NSC5AU2ZV3IE',
 'arn': 'arn:aws:iam::123837392027:user/bert-jan',
 'accountId': '123837392027',
 'accessKeyId': 'AKIATFQR7NSC8Q4X20BJ',
 'userName': 'bert-jan'}

## Flatten userIdentity

In [9]:
# The column userIdentity contains a dictionary like:

# {
#  'type': 'IAMUser',
#  'principalId': 'AID...',
#  'arn': 'arn:aws:iam::...',
#  'accountId': '...',
#  'accessKeyId': '...',
#  'userName': 'benjamin',
#  'sessionContext': {...},
#  'invokedBy': 'AWS Internal'
# }

# Dictionary is a nested column. Machine learning models cannot use dictionaries, so we need to flatten them into normal columns.

In [10]:
# Convert nested JSON to columns
user_identity_df = pd.json_normalize(df_identity['userIdentity'])

# Show the new columns
print(user_identity_df.columns)

Index(['type', 'principalId', 'arn', 'accountId', 'accessKeyId', 'userName',
       'invokedBy', 'sessionContext.attributes.creationDate',
       'sessionContext.attributes.mfaAuthenticated',
       'sessionContext.sessionIssuer.type',
       'sessionContext.sessionIssuer.principalId',
       'sessionContext.sessionIssuer.arn',
       'sessionContext.sessionIssuer.accountId',
       'sessionContext.sessionIssuer.userName',
       'sessionContext.ec2RoleDelivery'],
      dtype='str')


In [11]:
df_identity = pd.concat([df_identity, user_identity_df], axis=1)

In [12]:
df_identity.drop(columns=['userIdentity'], inplace=True)

In [13]:
df_identity.shape

(1000, 41)

In [14]:
df_identity[['userName','type','invokedBy']].head()

,userName,type,invokedBy
0,bert-jan,IAMUser,NaN
1,bert-jan,IAMUser,NaN
2,bert-jan,IAMUser,NaN
3,bert-jan,IAMUser,NaN
4,bert-jan,IAMUser,NaN


In [15]:
df_identity.columns

Index(['eventVersion', 'eventTime', 'eventSource', 'eventName', 'awsRegion',
       'sourceIPAddress', 'userAgent', 'requestParameters', 'responseElements',
       'requestID', 'eventID', 'readOnly', 'eventType', 'managementEvent',
       'recipientAccountId', 'eventCategory', 'tlsDetails', 'resources',
       'errorCode', 'errorMessage', 'additionalEventData', 'sharedEventID',
       'apiVersion', 'vpcEndpointId', 'sessionCredentialFromConsole',
       'serviceEventDetails', 'type', 'principalId', 'arn', 'accountId',
       'accessKeyId', 'userName', 'invokedBy',
       'sessionContext.attributes.creationDate',
       'sessionContext.attributes.mfaAuthenticated',
       'sessionContext.sessionIssuer.type',
       'sessionContext.sessionIssuer.principalId',
       'sessionContext.sessionIssuer.arn',
       'sessionContext.sessionIssuer.accountId',
       'sessionContext.sessionIssuer.userName',
       'sessionContext.ec2RoleDelivery'],
      dtype='str')

## Handle Missing Values in Identity Fields

In [16]:
missing = df_identity.isnull().sum().sort_values(ascending=False)
missing_percent = (missing / len(df_identity)) * 100

missing_table = pd.concat([missing, missing_percent], axis=1)
missing_table.columns = ["Missing Count", "Missing %"]

missing_table.head(15)

,Missing Count,Missing %
serviceEventDetails,999,99.9
apiVersion,999,99.9
sessionContext.ec2RoleDelivery,990,99.0
sharedEventID,986,98.6
sessionContext.sessionIssuer.userName,943,94.3
sessionContext.sessionIssuer.accountId,943,94.3
sessionContext.sessionIssuer.arn,943,94.3
sessionContext.sessionIssuer.principalId,943,94.3
sessionContext.sessionIssuer.type,943,94.3
vpcEndpointId,939,93.9


In [17]:
identity_cols = [
    'type',
    'principalId',
    'accountId',
    'accessKeyId',
    'userName',
    'invokedBy',
    'sessionContext.attributes.mfaAuthenticated',
    'sessionContext.sessionIssuer.type',
    'sessionContext.sessionIssuer.userName'
]

df_identity[identity_cols].isnull().sum()

type                                            1
principalId                                    14
accountId                                      13
accessKeyId                                    16
userName                                       71
invokedBy                                     870
sessionContext.attributes.mfaAuthenticated    746
sessionContext.sessionIssuer.type             943
sessionContext.sessionIssuer.userName         943
dtype: int64

In [18]:
# Create new col only for identity field - to avoid noise

identity_cols = [
    'type',
    'principalId',
    'accountId',
    'accessKeyId',
    'userName',
    'invokedBy',
    'sessionContext.attributes.mfaAuthenticated',
    'sessionContext.sessionIssuer.type',
    'sessionContext.sessionIssuer.userName'
]

df_identity[identity_cols].isnull().sum()

type                                            1
principalId                                    14
accountId                                      13
accessKeyId                                    16
userName                                       71
invokedBy                                     870
sessionContext.attributes.mfaAuthenticated    746
sessionContext.sessionIssuer.type             943
sessionContext.sessionIssuer.userName         943
dtype: int64

In [19]:
df_identity[identity_cols] = df_identity[identity_cols].fillna("Unknown")

In [20]:
df_identity[identity_cols].isnull().sum()

type                                          0
principalId                                   0
accountId                                     0
accessKeyId                                   0
userName                                      0
invokedBy                                     0
sessionContext.attributes.mfaAuthenticated    0
sessionContext.sessionIssuer.type             0
sessionContext.sessionIssuer.userName         0
dtype: int64

When detecting anomalies in cloud access logs, missing values can represent important behaviors. Replacing NaN with "Unknown" preserves the signal instead of deleting it.

In [21]:
missing_table.head(10)

,Missing Count,Missing %
serviceEventDetails,999,99.9
apiVersion,999,99.9
sessionContext.ec2RoleDelivery,990,99.0
sharedEventID,986,98.6
sessionContext.sessionIssuer.userName,943,94.3
sessionContext.sessionIssuer.accountId,943,94.3
sessionContext.sessionIssuer.arn,943,94.3
sessionContext.sessionIssuer.principalId,943,94.3
sessionContext.sessionIssuer.type,943,94.3
vpcEndpointId,939,93.9


In [22]:
df_identity[identity_cols].isnull().sum()

type                                          0
principalId                                   0
accountId                                     0
accessKeyId                                   0
userName                                      0
invokedBy                                     0
sessionContext.attributes.mfaAuthenticated    0
sessionContext.sessionIssuer.type             0
sessionContext.sessionIssuer.userName         0
dtype: int64

## Extract Time Features from eventTime

In [23]:
# Attackers often behave differently in time patterns:

# Examples:
# Access at 03:00 AM
# Access during weekends
# Unusual burst activity

# We will convert eventTime into useful features.

In [24]:
# Convert to date and time

df_identity['eventTime'] = pd.to_datetime(df_identity['eventTime'])
df_identity['eventTime'].head(1000)

0     2023-07-10 12:06:32+00:00
1     2023-07-10 12:06:32+00:00
2     2023-07-10 12:06:32+00:00
3     2023-07-10 12:06:32+00:00
4     2023-07-10 12:06:35+00:00
                 ...           
995   2023-07-10 11:57:49+00:00
996   2023-07-10 11:57:48+00:00
997   2023-07-10 11:57:49+00:00
998   2023-07-10 11:57:49+00:00
999   2023-07-10 11:57:48+00:00
Name: eventTime, Length: 1000, dtype: datetime64[us, UTC]

In [25]:
# Extract hour

df_identity['event_hour'] = df_identity['eventTime'].dt.hour
df_identity['event_hour'].head()

0    12
1    12
2    12
3    12
4    12
Name: event_hour, dtype: int32

In [26]:
# Extract day of the week

df_identity['event_dayofweek'] = df_identity['eventTime'].dt.dayofweek
df_identity['event_dayofweek'].head()

0    0
1    0
2    0
3    0
4    0
Name: event_dayofweek, dtype: int32

In [27]:
# Extract date

df_identity['event_date'] = df_identity['eventTime'].dt.date
df_identity['event_date'].head()

0    2023-07-10
1    2023-07-10
2    2023-07-10
3    2023-07-10
4    2023-07-10
Name: event_date, dtype: object

In [28]:
# Detect Weekend Activity

df_identity['is_weekend'] = df_identity['event_dayofweek'].isin([5,6]).astype(int)
df_identity['is_weekend'].head()

0    0
1    0
2    0
3    0
4    0
Name: is_weekend, dtype: int64